# Contrail segmentation — U-Net + EfficientNet-B2 · Colab V3

**Dataset:** GVCCS · Zenodo 15743988 · CC BY 4.0 · EUROCONTROL MUAC  
**Model:** U-Net + EfficientNet-B2 (7M params)  
**Resumes from:** HuggingFace Hub checkpoint (epoch 28, val Dice 0.7910)  

### Before running
1. Runtime → Change runtime type → **T4 GPU**
2. Left panel → 🔑 Secrets → add secret `HF_TOKEN` (your HuggingFace write token)  
   _(Settings → Access Tokens → New Token → Write)_
3. In Cell 3: set `HF_REPO = 'janisdombr/coav-contrail-checkpoints'`
4. **Runtime → Run all**
5. When prompted — allow Google Drive access (GVCCS stored there, not re-downloaded)

### What's different from Kaggle V2
| | Kaggle V2 | Colab V3 |
|---|---|---|
| Secrets | `kaggle_secrets` | `google.colab.userdata` |
| Dataset storage | `/kaggle/working` (resets) | **Google Drive** (persistent) |
| Val split | Last 10% (biased) | **Shuffled seed=42** (fixed) |
| Checkpoint | HF Hub after every epoch | HF Hub after every epoch |

In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import torch

N_GPU = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}  |  GPUs available: {N_GPU}')
for i in range(N_GPU):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')

if N_GPU == 0:
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU.')

In [ ]:
# ── 2. Google Drive + pip install ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/coav', exist_ok=True)

!pip install segmentation-models-pytorch albumentations huggingface_hub -q
print('Done.')

In [ ]:
# ── 3. Config & paths ─────────────────────────────────────────────────────────
import os

ENCODER    = 'efficientnet-b2'
EPOCHS     = 60     # warm restart: continue from ep 38 to 60 (22 new epochs)
BATCH_SIZE = 8
IMG_SIZE   = 512
START_LR   = 1e-4   # warm restart — fresh LR cycle from high value
WR_EPOCHS  = 22     # length of new cosine cycle (60 - 38 = 22)

HF_REPO       = 'janisdombr/coav-contrail-checkpoints'
HF_PUSH_EVERY = 1

DRIVE_DIR = '/content/drive/MyDrive/coav'
DATA_DIR  = f'{DRIVE_DIR}/GVCCS/train'
SAVE_DIR  = DRIVE_DIR
WORK_DIR  = '/content/coav'
os.makedirs(WORK_DIR, exist_ok=True)

PRETRAIN_CKPT = None
drive_ckpt = f'{DRIVE_DIR}/checkpoint_last.pt'
if os.path.exists(drive_ckpt):
    PRETRAIN_CKPT = drive_ckpt
    print(f'Local Drive checkpoint found: {PRETRAIN_CKPT}')
else:
    print('No Drive checkpoint — will restore from HuggingFace in next cell')

print(f'Encoder: {ENCODER}  |  Batch: {BATCH_SIZE}  |  Epochs: {EPOCHS}')
print(f'Warm restart: LR={START_LR:.1e}  cycle={WR_EPOCHS} epochs')

In [ ]:
# ── 3b. HuggingFace Hub setup ─────────────────────────────────────────────────
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = userdata.get('HF_TOKEN')
hf_api   = HfApi(token=HF_TOKEN)
hf_api.create_repo(repo_id=HF_REPO, repo_type='model', private=True, exist_ok=True)
print(f'HF repo ready → https://huggingface.co/{HF_REPO}')


def push_to_hf(is_best=False):
    files = ['checkpoint_last.pt']
    if is_best:
        files.append('contrail_unet_best.pt')
    for fname in files:
        path = f'{SAVE_DIR}/{fname}'
        if os.path.exists(path):
            hf_api.upload_file(
                path_or_fileobj=path,
                path_in_repo=fname,
                repo_id=HF_REPO,
                repo_type='model',
            )
    print(f'[HF] Pushed: {", ".join(files)}')


# Restore from HF if no Drive checkpoint
if PRETRAIN_CKPT is None:
    try:
        local = hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoint_last.pt',
            repo_type='model',
            token=HF_TOKEN,
            local_dir=DRIVE_DIR,
        )
        PRETRAIN_CKPT = local
        print('[HF] Checkpoint restored from HuggingFace → resuming training')
    except Exception as e:
        print(f'[HF] No checkpoint on HF — fresh start  ({e})')

In [ ]:
# ── 4. Download GVCCS to Google Drive (~2.1 GB, once only) ───────────────────
import json

if not os.path.exists(DATA_DIR):
    print('Downloading GVCCS from Zenodo to Google Drive (~2.1 GB)...')
    !wget -q --show-progress \
        'https://zenodo.org/records/15743988/files/GVCCS.zip?download=1' \
        -O /content/GVCCS.zip
    print('Extracting to Drive...')
    !unzip -q /content/GVCCS.zip -d {DRIVE_DIR}/
    !rm /content/GVCCS.zip
    print('Done — GVCCS saved to Google Drive (will not re-download next session).')
else:
    print('GVCCS already on Google Drive.')

with open(f'{DATA_DIR}/annotations.json') as f:
    _meta = json.load(f)
print(f'Images: {len(_meta["images"])}  |  Annotations: {len(_meta["annotations"])}')

In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
import json, cv2, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class GVCCSDataset(Dataset):
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir  = Path(data_dir)
        self.transform = transform
        with open(self.data_dir / 'annotations.json') as f:
            coco = json.load(f)
        self.ann_by_img = {}
        for ann in coco['annotations']:
            self.ann_by_img.setdefault(ann['image_id'], []).append(ann)
        imgs = coco['images']
        self.images = [imgs[i] for i in indices] if indices is not None else imgs

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        meta  = self.images[idx]
        image = cv2.cvtColor(
            cv2.imread(str(self.data_dir / 'images' / meta['file_name'])),
            cv2.COLOR_BGR2RGB)
        h, w  = meta.get('height', 1024), meta.get('width', 1024)
        mask  = np.zeros((h, w), dtype=np.uint8)
        for ann in self.ann_by_img.get(meta['id'], []):
            for seg in ann.get('segmentation', []):
                cv2.fillPoly(mask, [np.array(seg, dtype=np.int32).reshape(-1, 2)], 1)
        if self.transform:
            r = self.transform(image=image, mask=mask)
            return r['image'], r['mask'].unsqueeze(0).float()
        return image, mask

print('GVCCSDataset ready.')

In [ ]:
# ── 6. Transforms & DataLoaders ───────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-45, 45), p=0.5),
    A.ElasticTransform(alpha=80, sigma=4.0, p=0.25),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.GaussNoise(p=0.2),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, 48),
                    hole_width_range=(8, 48), p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

with open(f'{DATA_DIR}/annotations.json') as f:
    _n = len(json.load(f)['images'])

# GVCCS images ordered by sequence — last 10% = only last ~12 sequences (biased).
# Shuffle with fixed seed → val images from ALL 122 sequences uniformly.
random.seed(42)
all_indices = list(range(_n))
random.shuffle(all_indices)
val_size  = int(_n * 0.1)
val_idx   = all_indices[:val_size]
train_idx = all_indices[val_size:]

# num_workers=2 for Colab (more causes issues on some runtimes)
train_loader = DataLoader(
    GVCCSDataset(DATA_DIR, train_tf, train_idx),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(
    GVCCSDataset(DATA_DIR, val_tf, val_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_loader.dataset)} images ({len(train_loader)} batches)')
print(f'Val:   {len(val_loader.dataset)} images ({len(val_loader)} batches) — shuffled seed=42')

In [ ]:
# ── 7. Model ──────────────────────────────────────────────────────────────────
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
)

start_epoch = 1
best_dice   = 0.0
history     = []

if PRETRAIN_CKPT:
    ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu')
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_dice   = ckpt.get('best_dice', 0.0)
        history     = ckpt.get('history', [])
        print(f'Resumed from epoch {start_epoch - 1}, best dice {best_dice:.4f}')
    else:
        print('Checkpoint format not recognised — starting from epoch 1')
else:
    print('Fresh start from ImageNet weights (epoch 1)')

model = model.cuda()
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Encoder: {ENCODER}  |  Parameters: {params:.1f}M  |  Start epoch: {start_epoch}')

In [ ]:
# ── 8. Loss & optimizer ───────────────────────────────────────────────────────
dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary', gamma=2.0, alpha=0.25)
loss_fn    = lambda p, t: dice_loss(p, t) + focal_loss(p, t)

optimizer = torch.optim.AdamW(model.parameters(), lr=START_LR, weight_decay=2e-4)

# Warm restart: fresh cosine cycle of length WR_EPOCHS starting at START_LR.
# No scheduler fast-forward — optimizer must begin at full LR to escape the plateau.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=WR_EPOCHS, eta_min=1e-6)

print(f'Loss: Dice + FocalLoss(gamma=2, alpha=0.25)')
print(f'Warm restart LR: {START_LR:.1e} → 1e-6 over {WR_EPOCHS} epochs')

In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
import time


def dice_score(logits, target, threshold=0.5):
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum()
    if union < 1:
        return 1.0
    return (2 * inter / (union + 1e-8)).item()


def save_checkpoint(epoch, is_best=False):
    ckpt = {
        'epoch':           epoch,
        'encoder':         ENCODER,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_dice':       best_dice,
        'history':         history,
    }
    # Save to Drive (persistent) + HF push handles the rest
    torch.save(ckpt, f'{SAVE_DIR}/checkpoint_last.pt')
    if is_best:
        torch.save(model.state_dict(), f'{SAVE_DIR}/contrail_unet_best.pt')


try:
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()

        model.train()
        tr_loss = tr_dice = 0.0
        for images, masks in tqdm(train_loader,
                                  desc=f'Ep {epoch:02d}/{EPOCHS} train', leave=False):
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            logits = model(images)
            loss   = loss_fn(logits, masks)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()
            tr_dice += dice_score(logits, masks)

        model.eval()
        vl_loss = vl_dice = 0.0
        with torch.no_grad():
            for images, masks in tqdm(val_loader,
                                      desc=f'Ep {epoch:02d}/{EPOCHS} val  ', leave=False):
                images, masks = images.cuda(), masks.cuda()
                logits = model(images)
                vl_loss += loss_fn(logits, masks).item()
                vl_dice += dice_score(logits, masks)

        scheduler.step()

        nb, nvb = len(train_loader), len(val_loader)
        ep_dice = vl_dice / nvb
        is_best = ep_dice > best_dice
        if is_best:
            best_dice = ep_dice

        history.append(dict(epoch=epoch,
                            tr_loss=tr_loss/nb, tr_dice=tr_dice/nb,
                            vl_loss=vl_loss/nvb, vl_dice=ep_dice))

        save_checkpoint(epoch, is_best=is_best)

        if epoch % HF_PUSH_EVERY == 0:
            try:
                push_to_hf(is_best=is_best)
            except Exception as hf_err:
                print(f'[HF] Push failed (training continues): {hf_err}')

        marker = '  ← best' if is_best else ''
        print(f'Ep {epoch:02d}/{EPOCHS}  '
              f'tr loss={tr_loss/nb:.4f} dice={tr_dice/nb:.4f}  '
              f'val loss={vl_loss/nvb:.4f} dice={ep_dice:.4f}  '
              f'{time.time()-t0:.0f}s{marker}')

except KeyboardInterrupt:
    print('\nInterrupted — saving checkpoint...')
    save_checkpoint(epoch)
    try:
        push_to_hf(is_best=False)
    except Exception as hf_err:
        print(f'[HF] Push on interrupt failed: {hf_err}')

print(f'\nDone. Best val Dice: {best_dice:.4f}')
print(f'Weights: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 10. Training curves ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = [r['epoch'] for r in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_x, [r['tr_loss'] for r in history], label='train')
ax1.plot(epochs_x, [r['vl_loss'] for r in history], label='val')
ax1.set_xlabel('Epoch'); ax1.set_title('Loss (Dice + Focal)'); ax1.legend()

ax2.plot(epochs_x, [r['tr_dice'] for r in history], label='train')
ax2.plot(epochs_x, [r['vl_dice'] for r in history], label='val')
ax2.axhline(0.60, color='orange', linestyle='--', label='demo threshold 0.60')
ax2.axhline(0.75, color='green',  linestyle='--', label='PoC threshold 0.75')
ax2.axhline(0.7910, color='blue', linestyle=':', alpha=0.7, label='best so far 0.7910 (ep28)')
ax2.set_xlabel('Epoch'); ax2.set_title('Dice score'); ax2.legend()

plt.suptitle(f'V2/V3 ({ENCODER}) — Best val Dice = {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves_v3.png', dpi=120)
plt.show()

In [ ]:
# ── 11. Visual predictions on 6 val images ────────────────────────────────────
import random

raw_model = model.module if hasattr(model, 'module') else model
raw_model.eval()

with open(f'{DATA_DIR}/annotations.json') as f:
    all_imgs = json.load(f)['images']

samples = random.sample(val_idx, 6)
fig, axes = plt.subplots(6, 3, figsize=(11, 22))

for row, idx in enumerate(samples):
    meta = all_imgs[idx]
    img  = cv2.cvtColor(
        cv2.imread(f'{DATA_DIR}/images/{meta["file_name"]}'),
        cv2.COLOR_BGR2RGB)
    inp  = val_tf(image=img)['image'].unsqueeze(0).cuda()
    with torch.no_grad():
        pred = torch.sigmoid(raw_model(inp))[0, 0].cpu().numpy()

    axes[row, 0].imshow(img);               axes[row, 0].set_title('Input')
    axes[row, 1].imshow(pred, cmap='hot');  axes[row, 1].set_title('Heatmap')
    axes[row, 2].imshow(pred > 0.5, cmap='gray'); axes[row, 2].set_title('Mask (t=0.5)')
    for ax in axes[row]: ax.axis('off')

plt.suptitle(f'V3 val samples — Best Dice {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sample_predictions_v3.png', dpi=100)
plt.show()

In [ ]:
# ── 12. Output summary ────────────────────────────────────────────────────────
print(f'Encoder:     {ENCODER}')
print(f'Best Dice:   {best_dice:.4f}  (V1 baseline: 0.4918)')
print(f'Improvement: {(best_dice - 0.4918):+.4f}')
print()
for fname in ['contrail_unet_best.pt', 'checkpoint_last.pt',
              'training_curves_v3.png', 'sample_predictions_v3.png']:
    path = f'{SAVE_DIR}/{fname}'
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    status = f'{size:.1f} MB' if size else 'MISSING'
    print(f'  {fname:<42} {status}')

print(f'''
Google Drive: {DRIVE_DIR}/
HuggingFace:  https://huggingface.co/{HF_REPO}

Download weights for edge deployment:
  contrail_unet_best.pt → edge-pi/python/weights/contrail_unet.pt

Thresholds:
  Dice > 0.60  → demo-ready
  Dice > 0.75  → production PoC  ✓ (achieved at epoch 28)
''')